In [1]:
# W12D2: Install packages for the ML API and tests

%pip install -q fastapi uvicorn scikit-learn pytest httpx ruff

print("W12D2 environment setup complete!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 28.9 MB/s eta 0:00:00
W12D2 environment setup complete!


In [2]:
# W12D2: Create a simple ML prediction API

from fastapi import FastAPI
from pydantic import BaseModel
from sklearn.linear_model import LogisticRegression

app = FastAPI(title="W12D2 ML Prediction API")

# Sample training data
X = [[0], [1], [2], [3], [4], [5]]
y = [0, 0, 0, 1, 1, 1]

# Train the model
model = LogisticRegression()
model.fit(X, y)


class PredictionInput(BaseModel):
    value: float


@app.get("/health")
def health():
    return {"status": "healthy"}


@app.post("/predict")
def predict(data: PredictionInput):
    prediction = int(model.predict([[data.value]])[0])
    return {
        "input": data.value,
        "prediction": prediction
    }


print("ML API created successfully!")
print("Endpoints: GET /health and POST /predict")

ML API created successfully!
Endpoints: GET /health and POST /predict


In [3]:
# W12D2: Check Docker availability in the Colab runtime

import shutil
import subprocess

docker_path = shutil.which("docker")

if docker_path:
    print("Docker command found:", docker_path)
    result = subprocess.run(
        ["docker", "version"],
        capture_output=True,
        text=True
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
else:
    print("Docker command is not available in this Colab runtime.")

Docker command is not available in this Colab runtime.


In [5]:
# W12D2: Test the trained ML model

test_values = [0, 2, 4, 5]

for value in test_values:
    prediction = int(model.predict([[value]])[0])
    print(f"Input: {value} -> Prediction: {prediction}")

print("\nModel testing completed successfully!")

Input: 0 -> Prediction: 0
Input: 2 -> Prediction: 0
Input: 4 -> Prediction: 1
Input: 5 -> Prediction: 1

Model testing completed successfully!


In [6]:
# W12D2: Create project files

from pathlib import Path

project_dir = Path("/content/W12D2_Docker_for_AI")
project_dir.mkdir(parents=True, exist_ok=True)

app_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
from sklearn.linear_model import LogisticRegression

app = FastAPI(title="W12D2 ML Prediction API")

X = [[0], [1], [2], [3], [4], [5]]
y = [0, 0, 0, 1, 1, 1]

model = LogisticRegression()
model.fit(X, y)


class PredictionInput(BaseModel):
    value: float


@app.get("/health")
def health():
    return {"status": "healthy"}


@app.post("/predict")
def predict(data: PredictionInput):
    prediction = int(model.predict([[data.value]])[0])
    return {"input": data.value, "prediction": prediction}
'''

(project_dir / "app.py").write_text(app_code.strip() + "\n")

print("Project folder:", project_dir)
print("Created app.py:", (project_dir / "app.py").exists())
print("API file created successfully!")

Project folder: /content/W12D2_Docker_for_AI
Created app.py: True
API file created successfully!


In [7]:
# W12D2: Create dependency files and Dockerfile

# Python packages needed to run the ML API
(project_dir / "requirements.txt").write_text(
    "fastapi\n"
    "uvicorn\n"
    "scikit-learn\n"
)

# Packages needed for testing and linting
(project_dir / "requirements-dev.txt").write_text(
    "pytest\n"
    "httpx\n"
    "ruff\n"
)

# Docker image instructions
dockerfile = """
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""

(project_dir / "Dockerfile").write_text(dockerfile.strip() + "\n")

print("Created requirements.txt")
print("Created requirements-dev.txt")
print("Created Dockerfile")
print("\nProject files:")
for file in sorted(project_dir.iterdir()):
    print("-", file.name)

Created requirements.txt
Created requirements-dev.txt
Created Dockerfile

Project files:
- Dockerfile
- app.py
- requirements-dev.txt
- requirements.txt


In [8]:
# W12D2: Create automated API tests

tests_dir = project_dir / "tests"
tests_dir.mkdir(parents=True, exist_ok=True)

test_code = '''
from fastapi.testclient import TestClient
from app import app

client = TestClient(app)


def test_health():
    response = client.get("/health")
    assert response.status_code == 200
    assert response.json()["status"] == "healthy"


def test_prediction():
    response = client.post("/predict", json={"value": 4})
    assert response.status_code == 200
    assert response.json()["prediction"] == 1
'''

(tests_dir / "test_app.py").write_text(test_code.strip() + "\n")

print("Created tests/test_app.py")
print("Health and prediction tests are ready!")

Created tests/test_app.py
Health and prediction tests are ready!


In [9]:
# W12D2: Create automated API tests

tests_dir = project_dir / "tests"
tests_dir.mkdir(parents=True, exist_ok=True)

test_code = '''
from fastapi.testclient import TestClient
from app import app

client = TestClient(app)


def test_health():
    response = client.get("/health")
    assert response.status_code == 200
    assert response.json()["status"] == "healthy"


def test_prediction():
    response = client.post("/predict", json={"value": 4})
    assert response.status_code == 200
    assert response.json()["prediction"] == 1
'''

(tests_dir / "test_app.py").write_text(test_code.strip() + "\n")

print("Created tests/test_app.py")
print("Health and prediction tests are ready!")

Created tests/test_app.py
Health and prediction tests are ready!


In [10]:
# Install development dependencies
%pip install -q pytest httpx ruff

# Run the API tests
%cd /content/W12D2_Docker_for_AI
!python -m pytest -v

/content/W12D2_Docker_for_AI
============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/W12D2_Docker_for_AI
plugins: platformdirs-4.12.2, anyio-4.15.1, langsmith-0.14.2, typeguard-4.6.0
collected 2 items                                                              

tests/test_app.py::test_health PASSED                                    [ 50%]
tests/test_app.py::test_prediction PASSED                                [100%]

============================== 2 passed in 1.68s ===============================


In [11]:
# W12D2: Document monitoring and retraining strategy

monitoring_text = """
# ML API Monitoring Strategy

## 1. Application Monitoring
- Monitor API health using the GET /health endpoint.
- Track prediction request counts and response times.
- Record HTTP errors and failed requests.
- Monitor CPU, memory, and container resource usage.

## 2. Alerts
- Alert when the API health check fails.
- Alert when HTTP 5xx errors increase.
- Alert when response latency exceeds the service target.
- Alert when CPU or memory usage remains unusually high.

## 3. Model Monitoring
- Track prediction distributions over time.
- Monitor changes in incoming feature distributions.
- Compare predictions with actual outcomes when labels become available.
- Check for data drift and model performance degradation.

## 4. Retraining Triggers
- Retrain when model performance falls below the agreed threshold.
- Investigate significant data or concept drift.
- Retrain when enough new, representative labelled data is available.
- Validate the new model before deployment.

## 5. Deployment Safety
- Run automated tests before building the Docker image.
- Use versioned images and model versions.
- Keep the previous working version available for rollback.
- Never deploy a newly trained model without validation.
"""

(project_dir / "monitoring.md").write_text(
    monitoring_text.strip() + "\n"
)

print("Created monitoring.md")
print((project_dir / "monitoring.md").read_text())

Created monitoring.md
# ML API Monitoring Strategy

## 1. Application Monitoring
- Monitor API health using the GET /health endpoint.
- Track prediction request counts and response times.
- Record HTTP errors and failed requests.
- Monitor CPU, memory, and container resource usage.

## 2. Alerts
- Alert when the API health check fails.
- Alert when HTTP 5xx errors increase.
- Alert when response latency exceeds the service target.
- Alert when CPU or memory usage remains unusually high.

## 3. Model Monitoring
- Track prediction distributions over time.
- Monitor changes in incoming feature distributions.
- Compare predictions with actual outcomes when labels become available.
- Check for data drift and model performance degradation.

## 4. Retraining Triggers
- Retrain when model performance falls below the agreed threshold.
- Investigate significant data or concept drift.
- Retrain when enough new, representative labelled data is available.
- Validate the new model before deployment.

In [12]:
# W12D2: Create GitHub Actions CI/CD workflow

github_dir = project_dir / ".github" / "workflows"
github_dir.mkdir(parents=True, exist_ok=True)

workflow = """
name: W12D2 ML API CI/CD

on:
  push:
    branches: ["**"]
  pull_request:
    branches: ["**"]

jobs:
  lint-test-build:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt -r requirements-dev.txt

      - name: Lint Python files
        run: ruff check app.py tests

      - name: Run automated tests
        run: pytest -v

      - name: Build Docker image
        run: docker build -t w12d2-ml-api:${{ github.sha }} .

      - name: Log in to Docker Hub
        if: github.event_name == 'push' && github.ref == 'refs/heads/main'
        uses: docker/login-action@v3
        with:
          username: ${{ secrets.DOCKERHUB_USERNAME }}
          password: ${{ secrets.DOCKERHUB_TOKEN }}

      - name: Push image to Docker Hub
        if: github.event_name == 'push' && github.ref == 'refs/heads/main'
        run: |
          docker tag w12d2-ml-api:${{ github.sha }} ${{ secrets.DOCKERHUB_USERNAME }}/w12d2-ml-api:${{ github.sha }}
          docker push ${{ secrets.DOCKERHUB_USERNAME }}/w12d2-ml-api:${{ github.sha }}
"""

workflow_path = github_dir / "ci.yml"
workflow_path.write_text(workflow.strip() + "\n")

print("Created:", workflow_path)
print("Workflow file exists:", workflow_path.exists())
print("\nWorkflow includes linting, tests, Docker build, and registry push.")

Created: /content/W12D2_Docker_for_AI/.github/workflows/ci.yml
Workflow file exists: True

Workflow includes linting, tests, Docker build, and registry push.


In [13]:
# W12D2: Save API response outputs

import json
from fastapi.testclient import TestClient

outputs_dir = project_dir / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

client = TestClient(app)

# Capture the actual health endpoint response
health_response = client.get("/health")
health_data = {
    "status_code": health_response.status_code,
    "response": health_response.json()
}

# Capture the actual prediction endpoint response
prediction_response = client.post(
    "/predict",
    json={"value": 4}
)
prediction_data = {
    "status_code": prediction_response.status_code,
    "response": prediction_response.json()
}

(outputs_dir / "health_output.json").write_text(
    json.dumps(health_data, indent=2) + "\n"
)
(outputs_dir / "prediction_output.json").write_text(
    json.dumps(prediction_data, indent=2) + "\n"
)

print("Health response:")
print(json.dumps(health_data, indent=2))

print("\nPrediction response:")
print(json.dumps(prediction_data, indent=2))

print("\nSaved both API outputs successfully!")

Health response:
{
  "status_code": 200,
  "response": {
    "status": "healthy"
  }
}

Prediction response:
{
  "status_code": 200,
  "response": {
    "input": 4.0,
    "prediction": 1
  }
}

Saved both API outputs successfully!


In [14]:
# W12D2: Run tests and save the output

import subprocess

result = subprocess.run(
    ["python", "-m", "pytest", "-v"],
    cwd=str(project_dir),
    capture_output=True,
    text=True
)

test_output = result.stdout + "\n" + result.stderr

(outputs_dir / "test_results.txt").write_text(test_output)

print(test_output)
print("Test exit code:", result.returncode)

if result.returncode != 0:
    raise RuntimeError("Tests failed. Fix the errors before continuing.")

print("Test results saved to outputs/test_results.txt")

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/W12D2_Docker_for_AI
plugins: platformdirs-4.12.2, anyio-4.15.1, langsmith-0.14.2, typeguard-4.6.0
collecting ... collected 2 items

tests/test_app.py::test_health PASSED                                    [ 50%]
tests/test_app.py::test_prediction PASSED                                [100%]

============================== 2 passed in 2.04s ===============================


Test exit code: 0
Test results saved to outputs/test_results.txt


In [15]:
# W12D2: Check Python code quality

import subprocess

lint_result = subprocess.run(
    ["python", "-m", "ruff", "check", "app.py", "tests"],
    cwd=str(project_dir),
    capture_output=True,
    text=True
)

print(lint_result.stdout or "No lint errors found.")
if lint_result.stderr:
    print(lint_result.stderr)

if lint_result.returncode != 0:
    print("Linting found issues. We will fix them before submission.")
else:
    print("SUCCESS: Ruff linting passed!")

I001 [*] Import block is un-sorted or un-formatted
 --> tests/test_app.py:1:1
  |
1 | / from fastapi.testclient import TestClient
2 | | from app import app
  | |___________________^
3 |
4 |   client = TestClient(app)
  |
help: Organize imports
  |
1 | from fastapi.testclient import TestClient
2 +
3 | from app import app
  |

Found 1 error.
[*] 1 fixable with `--fix`.

Linting found issues. We will fix them before submission.


In [16]:
# W12D2: Check Docker availability

import shutil
import subprocess

docker_path = shutil.which("docker")

if docker_path:
    print("Docker command found:", docker_path)

    result = subprocess.run(
        ["docker", "version"],
        capture_output=True,
        text=True,
        timeout=20
    )

    print(result.stdout)
    if result.stderr:
        print(result.stderr)

    if result.returncode == 0:
        print("Docker is available. We can try building the image.")
    else:
        print("Docker is installed but the daemon may not be available.")
else:
    print("Docker is not available in this Colab runtime.")
    print("The GitHub Actions workflow will build the image on its runner.")

Docker is not available in this Colab runtime.
The GitHub Actions workflow will build the image on its runner.


In [17]:
# W12D2: List files for GitHub submission

for path in sorted(project_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(project_dir))

.github/workflows/ci.yml
.pytest_cache/.gitignore
.pytest_cache/CACHEDIR.TAG
.pytest_cache/README.md
.pytest_cache/v/cache/nodeids
.ruff_cache/.gitignore
.ruff_cache/0.17.1/15124375630422524551
.ruff_cache/0.17.1/2638924133377936707
.ruff_cache/CACHEDIR.TAG
Dockerfile
__pycache__/app.cpython-313.pyc
app.py
monitoring.md
outputs/health_output.json
outputs/prediction_output.json
outputs/test_results.txt
requirements-dev.txt
requirements.txt
tests/__pycache__/test_app.cpython-313-pytest-8.4.2.pyc
tests/test_app.py


In [18]:
# W12D2: Download individual project files

from google.colab import files
from pathlib import Path
import time

files_to_download = [
    "app.py",
    "Dockerfile",
    "requirements.txt",
    "requirements-dev.txt",
    "README.md",
    "monitoring.md",
    ".gitignore",
    "tests/test_app.py",
    ".github/workflows/ci.yml",
    "outputs/health_output.json",
    "outputs/prediction_output.json",
    "outputs/test_results.txt",
]

for name in files_to_download:
    file_path = project_dir / name

    if file_path.exists():
        print("Downloading:", name)
        files.download(str(file_path))
        time.sleep(2)
    else:
        print("Missing file:", name)

print("Download requests completed!")

Downloading: app.py


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: Dockerfile


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Downloading: outputs/test_results.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download requests completed!
